# Notebook 02: Real data: load every sample into one object per cohort

**Goal.** Read the downloaded count files (any common format) and save one raw dataset per cohort. **Skip in the dry run.**

> REAL DATA ONLY - skip this notebook in the dry run.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Reads the sample sheet (eligible rows only) and shows the files of the first sample so you can confirm the paths are right before loading everything.

In [ ]:
from mlc.dataset import read_any, concat, inventory
sheet = pd.read_csv(SHEET)
sheet = sheet[sheet.eligible_primary.astype(str).str.lower() == "true"]
inventory(sheet.files.iloc[0])           # sanity: what is in the first sample's folder?

**Cell 3.** **read_any** detects the format (10x triplet, dense table, h5ad) automatically. Cell names are prefixed with the sample ID so two samples can never share a cell name. One combined dataset per cohort is saved as *{cohort}_raw*, exactly the name the simulated data used, so notebooks 03 onwards run unchanged.

In [ ]:
for c, sub in sheet.groupby("cohort"):
    parts = []
    for _, r in sub.iterrows():
        pre = r["prefix"] if "prefix" in sub.columns and isinstance(r["prefix"], str) else None
        d = read_any(r["files"], prefix=pre)
        d.obs.index = [f"{r.sample_id}_{i}" for i in d.obs.index]
        d.obs["sample_id"], d.obs["cohort"], d.obs["age_months"] = r.sample_id, c, float(r.age_months)
        parts.append(d); print(r.sample_id, d.n_obs, "cells")
    ds = concat(parts, join="outer"); ds.save(PROC / f"{c}_raw"); print(c, ds)